In [ ]:
# Standard library
import json
import os
import uuid

# Third-party core
import boto3
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# SageMaker
import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker.model import Model
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join, JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import CacheConfig, ProcessingStep, TrainingStep

sns.set_theme()

In [ ]:
%store -r
%store

In [ ]:
# Reassigning store variables for IDE convenience
bucket = bucket
database_name = database_name
project_prefix = project_prefix
landmarks_table = landmarks_table
s3_staging_dir = s3_staging_dir
s3_raw_cats_prefix = s3_raw_cats_prefix
manifest_table = manifest_table

In [ ]:
s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)


# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

s3_athena_results_dir = f"s3://{bucket}/{project_prefix}/athena/results/"

In [ ]:
%store s3_athena_results_dir

# Building Datasets for Modelling

In [ ]:
statement_image_manifest = f"""
	select *
	from
		{database_name}.{manifest_table}
"""

image_manifest_df = pd.read_sql(statement_image_manifest, engine)
image_manifest_df.head()

In [ ]:
# Classifier data: original images from every source, with the splits assigned at ingestion.
# Augmented cat images (v2/v3) exist only for the positive class, so they are left out:
# their flips, noise and blur would otherwise become a shortcut for "cat".
training_manifest_features = ["remote_path", "label", "split", "source"]
training_manifest = image_manifest_df.loc[image_manifest_df["version"] == "raw", training_manifest_features]

training_manifest.groupby(["source", "label", "split"]).size().unstack("split")

In [ ]:
training_uuid = uuid.uuid4()
training_uuid

In [ ]:
os.makedirs("data/processed/combined", exist_ok=True)

# Saving local copy of training manifest
training_manifest_local_path = f"data/processed/combined/training_manifest_{training_uuid}.pqt"
training_manifest.to_parquet(training_manifest_local_path)

# Saving training manifest to S3
training_manifest_remote_path = f"{project_prefix}/data/processed/combined/training_manifests/training_manifest_{training_uuid}.pqt"
s3.upload_file(training_manifest_local_path, bucket, training_manifest_remote_path)

training_manifest_s3_uri = f"s3://{bucket}/{training_manifest_remote_path}"

%store training_manifest_remote_path

# Pipeline Parameters

In [ ]:
# caching
cache_config = CacheConfig(enable_caching=True, expire_after="PT1H")

# Data: the training manifest built above (overridable per execution)
input_manifest_uri = ParameterString(
    name="InputManifestUri", default_value=training_manifest_s3_uri
    )

# Registry
model_package_group_name = "CatClassifierModelPackageGroup"

model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="Approved"
)

# Quality gate: F1 for the cat class on the held-out test split
f1_threshold = ParameterFloat(name="F1Threshold", default_value=0.90)

# Instances
processing_instance_type = "ml.m5.xlarge"
training_instance_type = "ml.g5.4xlarge"
inference_instance_type = "ml.g5.xlarge"

# Preprocessing Step

In [ ]:
pipeline_session = PipelineSession()

# CPU image: preprocessing and evaluation do not need a GPU
processing_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type=processing_instance_type,
    image_scope="training",
)

def make_processor(base_job_name):
    return ScriptProcessor(
        command=["python3"],
        image_uri=processing_image_uri,
        role=role,
        instance_type=processing_instance_type,
        instance_count=1,
        base_job_name=base_job_name,
        sagemaker_session=pipeline_session,
    )

# One step per split: download images listed in the manifest, resize to 224x224, write labels
def make_preprocessing_step(split):
    step_args = make_processor(f"cls-preprocess-{split}").run(
        code="src/classification/preprocess.py",
        inputs=[
            ProcessingInput(
                source=input_manifest_uri,
                destination="/opt/ml/processing/manifest"
            )
        ],
        outputs=[
            ProcessingOutput(
                output_name=split,
                source="/opt/ml/processing/output"
            )
        ],
        arguments=[
            "--manifest", "/opt/ml/processing/manifest",
            "--output",   "/opt/ml/processing/output",
            "--split",    split,
        ],
    )
    return ProcessingStep(
        name=f"ClassifierPreprocessing{split.title()}",
        step_args=step_args,
        cache_config=cache_config,
    )

step_process_train = make_preprocessing_step("train")
step_process_val = make_preprocessing_step("validation")
step_process_test = make_preprocessing_step("test")

# Training Step

In [ ]:
model_path = f"s3://{bucket}/{project_prefix}/models/classifier"

pytorch_train = PyTorch(
    entry_point="train.py",
    source_dir="./src/classification",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type=training_instance_type,
    instance_count=1,
    output_path=model_path,
    sagemaker_session=pipeline_session,
    hyperparameters={
        "epochs": 4,
        "learning-rate": 0.001,
        "batch-size": 64,
    },
    # Per-epoch metrics printed by train.py, recorded on the training job
    metric_definitions=[
        {"Name": "train:loss", "Regex": "train_loss=([0-9\\.]+)"},
        {"Name": "train:accuracy", "Regex": "train_acc=([0-9\\.]+)"},
        {"Name": "validation:loss", "Regex": "val_loss=([0-9\\.]+)"},
        {"Name": "validation:accuracy", "Regex": "val_acc=([0-9\\.]+)"},
    ],
)

train_args = pytorch_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process_train.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
        ),
        "validation": TrainingInput(
            s3_data=step_process_val.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
        ),
    }
)

step_train = TrainingStep(
    name="TrainClassifierModel",
    step_args=train_args,
    cache_config=cache_config
)

# Evaluation Step

In [ ]:
eval_args = make_processor("cls-eval").run(
    code="src/classification/evaluation.py",
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process_test.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
        # Training labels and image statistics only (not the images), for the benchmarks
        ProcessingInput(
            source=Join(on="/", values=[
                step_process_train.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
                "labels.parquet",
            ]),
            destination="/opt/ml/processing/train",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation"
        ),
    ],
    arguments=["--batch-size", "64"],
)

evaluation_report = PropertyFile(
    name="ClassifierEvaluationReport", output_name="evaluation", path="evaluation.json"
)

step_eval = ProcessingStep(
    name="ClassifierModelEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

# Registry Step

In [ ]:
# Serving image. train.py packages src/classification/inference.py into model.tar.gz
# under code/, so the registered model deploys straight from the registry.
inference_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type=inference_instance_type,
    image_scope="inference",
)

model = Model(
    image_uri=inference_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    env={
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    },
    sagemaker_session=pipeline_session,
    role=role,
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["image/jpeg", "image/png", "application/x-image", "application/x-npy"],
    response_types=["application/json"],
    inference_instances=[inference_instance_type],
    transform_instances=[inference_instance_type],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
    description="ResNet-18 cat vs. non-cat image classifier",
)

step_register = ModelStep(name="ClassifierRegisterModel", step_args=register_args)

# Fail Step

In [ ]:
step_fail = FailStep(
    name="ClassifierF1Fail",
    error_message=Join(on=" ", values=["Execution failed due to F1 <", f1_threshold]),
)

# Condition Step

In [ ]:
cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="binary_classification_metrics.f1.value",
    ),
    right=f1_threshold,
)

step_cond = ConditionStep(
    name="ClassifierF1Cond",
    conditions=[cond_gte],
    if_steps=[step_register],
    else_steps=[step_fail],
)

# Pipeline Execution

In [ ]:
pipeline = Pipeline(
    name="CatClassifierPipeline",
    parameters=[input_manifest_uri, model_approval_status, f1_threshold],
    steps=[step_process_train, step_process_val, step_process_test, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

In [ ]:
pipeline.upsert(role_arn=role)
execution = pipeline.start()

# Preprocessing downloads every image in the manifest, so allow up to 2 hours
execution.wait(delay=60, max_attempts=120)
print(execution.describe()["PipelineExecutionStatus"])

In [ ]:
for step in execution.list_steps():
    print(f"Step: {step['StepName']}")
    print(f"Status: {step['StepStatus']}")

    if step['StepStatus'] == 'Failed':
        print(f"Failure Reason: {step['FailureReason']}")
    print("---")

# Pipeline Outputs

In [ ]:
# Resolve this execution's outputs for 6_Monitoring
steps = {step["StepName"]: step for step in execution.list_steps()}

# Registered (condition-approved) model package
cls_model_package_arn = steps["ClassifierRegisterModel-RegisterModel"]["Metadata"]["RegisterModel"]["Arn"]
cls_model_package = sagemaker_client.describe_model_package(ModelPackageName=cls_model_package_arn)
cls_model_data_url = cls_model_package["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]

# Evaluation report attached to the registered model
evaluation_uri = cls_model_package["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
evaluation_bucket, evaluation_key = evaluation_uri.replace("s3://", "").split("/", 1)
cls_evaluation = json.loads(s3.get_object(Bucket=evaluation_bucket, Key=evaluation_key)["Body"].read())

print("Model package:", cls_model_package_arn)
print("Model data:", cls_model_data_url)

%store cls_model_package_arn
%store cls_model_data_url

# Test Set Evaluation

In [ ]:
metrics = cls_evaluation["binary_classification_metrics"]
print(f"Test samples: {cls_evaluation['test_samples']}")

# Binary metrics treat cat (label 1) as the positive class
display(pd.DataFrame(
    {name: [metrics[name]["value"]] for name in ["accuracy", "precision", "recall", "f1", "auc", "true_negative_rate"]},
    index=["test"],
).round(4))

display(pd.DataFrame(cls_evaluation["per_class_metrics"]).T.round(4))

# Per-source results. Each source contributes both classes except kaggle_cats (cats only),
# so a model that learned image source instead of content shows up as a weak source here.
pd.DataFrame(cls_evaluation["per_source_metrics"]).T.round(4)

In [ ]:
cm = metrics["confusion_matrix"]
cm_df = pd.DataFrame(
    [[cm["0"]["0"], cm["0"]["1"]], [cm["1"]["0"], cm["1"]["1"]]],
    index=["actual non-cat", "actual cat"],
    columns=["predicted non-cat", "predicted cat"],
)

fig, ax = plt.subplots()
sns.heatmap(cm_df, annot=True, fmt="d", cmap="crest", ax=ax)
ax.set_title("Classifier confusion matrix (test split)")
plt.tight_layout()
plt.show()

# Benchmarks
Two benchmarks, scored on the same test split:
- **Majority class** always predicts the more common class in training. Any useful model has to beat it.
- **Image statistics** is a logistic regression trained on width, height, aspect ratio, brightness, contrast and file size of each original image. None of these describe what is in the picture. A high score means the classes can be told apart by image source or format alone, which is the shortcut the multi-source data is meant to remove. It is a linear model, so a low score is reassuring but does not rule out every shortcut a deep network could find.

In [ ]:
benchmarks = cls_evaluation["benchmarks"]
metric_names = ["accuracy", "precision", "recall", "f1", "auc", "true_negative_rate"]

display(pd.DataFrame({
    "model": {name: metrics[name]["value"] for name in metric_names},
    f"majority class ({benchmarks['majority_class']['predicts']})": {name: benchmarks["majority_class"][name] for name in metric_names},
    "image statistics": {name: benchmarks["image_statistics"][name] for name in metric_names},
}).round(4))

# Where the image statistics alone separate the classes
pd.DataFrame({
    "model_accuracy": {source: m["accuracy"] for source, m in cls_evaluation["per_source_metrics"].items()},
    "image_statistics_accuracy": {source: m["accuracy"] for source, m in benchmarks["image_statistics"]["per_source_metrics"].items()},
}).round(4)